# Extracting ITS_LIVE velocity data 

## Packages to install (needs to be done via conda activate in terminal):
- zarr 
- pyproj
- s3fs

## In terminal:
The version needs to be specified so the versions don't conflict with each other, run this in a new terminal (in VS code), once the packages have been installed in conda:
"pip install "zarr<3" "fsspec==2025.3.0" s3fs xarray pyproj" 

## Script info: 
This notebook is in here for reference, but it may not run in VSCode. In order to extract the velocity maps, it needed AWS access which requires admin approval. I got around it by running in Colab since their server already has access. Additionally, if you do want to run it in colab, it is extremely slow, it took about 4 hours to finsih running. 

## Other data: 
- I emailed you guys my tidal data set, the code below is set up for it to like in the data folder of our project folder. We can dicuss where to host it later. Right now, those files are in the git ignore 

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import s3fs
import zarr
from pyproj import Transformer

# Double check proper zarr version 
print("zarr version:", zarr.__version__)

#Location of GPS stations the tidal features are from 
# Based on Zachary's zendo --- will link in metadata or README eventually
GPS_stations = {
    "WB07": {"lat": -83.5200, "lon": -138.0800},
    "WB08": {"lat": -83.4650, "lon": -138.7500},
    "WB09": {"lat": -83.6200, "lon": -137.4000},
    "WB10": {"lat": -83.5800, "lon": -138.3500},
    "WB11": {"lat": -83.4100, "lon": -139.3000},
    "WB12": {"lat": -83.6700, "lon": -139.1000},
    "WB13": {"lat": -83.5500, "lon": -137.8000},
    "WB14": {"lat": -83.4900, "lon": -140.0000},
    "ENB":  {"lat": -83.4200, "lon": -137.7200},
    "SLW":  {"lat": -83.6380, "lon": -138.1700},
}

# Read in Kaitlyn's icequake data file (the velocity data will go with this)
#sort based on time and check it 
df = pd.read_csv("../Data/Whillians-GPS-Data-and-Features.csv", parse_dates=["start_time"])
df = df.sort_values("start_time").reset_index(drop=True)
print(f"Catalog: {len(df):,} events  {df['start_time'].min().date()} → {df['start_time'].max().date()}")

#Open and sort datacube (datacube of ITS_LIVE)
ZARR_S3 = "s3://its-live-data/datacubes/v2/S80W140/ITS_LIVE_vel_EPSG3031_G0120_X-450000_Y-550000.zarr"
fs    = s3fs.S3FileSystem(anon=True)
store = s3fs.S3Map(root=ZARR_S3, s3=fs, check=False)
ds    = xr.open_zarr(store, consolidated=True, decode_timedelta=False)
ds    = ds.sortby("mid_date")
print(f"Datacube: {str(ds.mid_date.values[0])[:10]} → {str(ds.mid_date.values[-1])[:10]}")
print(f"Total epochs: {len(ds.mid_date)}")

#Project to the coordinates used for Antarctica
tr = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True)
station_xy = {
    name: tr.transform(coords["lon"], coords["lat"])
    for name, coords in GPS_stations.items()
}

# Get the nearest velocity chunk for every event in the catalog
records = []
for station, (sx, sy) in station_xy.items():
    print(f"  {station}...")
    ds_pt = ds.sel(x=sx, y=sy, method="nearest")
    for _, event in df.iterrows():
        ds_t = ds_pt.sel(mid_date=event["start_time"], method="nearest")
        records.append({
            "station"    : station,
            "start_time" : event["start_time"],
            "time_since" : event["time_since"],
            "slip_size"  : event["slip_size"],
            "tide_height": event["tide_height"],
            "high_t_evt" : event["high_t_evt"],
            "v_myr"      : float(ds_t["v"].values)  if "v"  in ds_t else np.nan,
            "vx_myr"     : float(ds_t["vx"].values) if "vx" in ds_t else np.nan,
            "vy_myr"     : float(ds_t["vy"].values) if "vy" in ds_t else np.nan,
        })
#Group the results together based on the GPS station 
merged = pd.DataFrame(records)
merged = merged.dropna(subset=["v_myr"]).reset_index(drop=True)
print(f"\nDone: {len(merged):,} rows  ({merged['station'].nunique()} stations)")
print(f"Unique v_myr values per station:")
for st, grp in merged.groupby("station"):
    print(f"  {st}: {grp['v_myr'].nunique()} unique values, mean={grp['v_myr'].mean():.1f} m/yr")

# Merge this with the tidal data set
merged.to_csv("whillans_velocity_slip_merged-new.csv", index=False)


zarr version: 3.1.6
Catalog: 5,150 events  2008-01-25 → 2019-11-23


Questions: 
- Where to host data?
- Not good enough temporal resolution. How do I fix?


To Check: 
- Make sure the new merged data set goes through the approraite preprocessing before ML methods, but not before standard data visualization 
- Make sure the way the grouping was done is correct 

Next: 
- Make sure it's cleaned 
- Run stats
- Do needed interpolation 